# Notebook 01: Parse mldd Catalog

## What this notebook does

This is the **first step** of the FM7 focal mechanism pipeline.  
Before we can compute focal mechanisms, we need to know *when* and *where* each earthquake happened, and *which seismic phases* (P and S waves) were picked at each station.

The `mldd` (Maximum Likelihood Double-Difference) relocator outputs two file types for each dataset year:

| File | Contents |
|------|-----------|
| `.loc` | One line per earthquake — origin time, latitude, longitude, depth, magnitude, event ID |
| `.pha` | Phase pick file — for each earthquake, lists each station and the P/S arrival time |

We have **two years** of data:
- **Y1**: Sep 2022 – Aug 2023 → **6,727 events**
- **Y2**: Sep 2023 – Sep 2024 → **48,244 events**

**What we do in this notebook:**
1. Parse both `.loc` files into a single Pandas DataFrame (`catalog`)
2. Parse both `.pha` files to extract P and S arrival times per station
3. Validate the data (count events, check geographic bounds)
4. Save both DataFrames as Parquet files (fast, compact binary format)
5. Make 4 diagnostic plots and send a completion email

**Why Parquet?** CSV files are slow to read and large. Parquet stores data column-by-column with compression, making it ~10x faster to load and ~5x smaller on disk. It also preserves data types (datetime, float32) exactly.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import sys
from pathlib import Path
from datetime import datetime

# ── Numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for long runs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# seaborn makes heatmaps easy; falls back gracefully if not installed
try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False
    print('seaborn not found — coverage heatmap will use matplotlib fallback')

print('Imports OK')
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

In [ ]:
# ── Add the python/ directory to sys.path so we can import our config files ──
# sys.path is the list of directories Python searches when you write `import something`.
# By inserting our scripts folder at position 0, our local files take priority.
PYTHON_DIR = Path('/Users/mczhang/Documents/GitHub/FM7/01-scripts/python')
sys.path.insert(0, str(PYTHON_DIR))

from pipeline_config import (
    # Input files
    LOC_Y1, LOC_Y2, PHA_Y1, PHA_Y2,
    # Output files
    CATALOG_PQ, PHASES_PQ,
    # Station definitions
    STATIONS_ALL, STATION_COORDS,
    # Email
    EMAIL_PLOTS_DIR,
    # Date labels
    Y1_START, Y1_END, Y2_START, Y2_END,
)
from email_utils import step_email

# Create output directories if they don't exist yet.
# parents=True means it also creates intermediate folders.
# exist_ok=True means no error if the folder already exists.
CATALOG_PQ.parent.mkdir(parents=True, exist_ok=True)
EMAIL_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Config loaded.')
print(f'  LOC_Y1 : {LOC_Y1}')
print(f'  LOC_Y2 : {LOC_Y2}')
print(f'  PHA_Y1 : {PHA_Y1}')
print(f'  PHA_Y2 : {PHA_Y2}')

## Step 1: Read Location Files (`.loc`)

Each `.loc` file has **one earthquake per line**, space-delimited, with these columns:

```
YY  MM  DD  HH  MIN  SS.sss  Lat  Lon  Depth  Mag  EventID
2022  9  1  9  35  49.830  45.955623  -129.990910  1.818  -1.39  224400000
```

A few things to notice:
- The date/time is split across **six separate columns**, so we have to assemble it into a Python `datetime` object manually.
- Seconds are stored as a float (e.g. `49.830`), so we split off the integer part and the fractional part separately.
- Longitude is **negative** (west of the prime meridian), which is the standard geographic convention.

We write a `parse_loc_file()` function that reads one file and returns a Pandas `DataFrame`.  
Then we call it twice (once for Y1, once for Y2), add a `'year'` column to each, and concatenate them.

In [ ]:
def parse_loc_file(filepath):
    """
    Read a mldd .loc file and return a DataFrame.

    Each line must have at least 11 space-separated tokens:
        YY MM DD HH MIN SS.sss Lat Lon Depth Mag EventID

    Returns
    -------
    pd.DataFrame with columns:
        event_id (int64), time (datetime64[us]), lat, lon,
        depth (km), mag (float32)
    """
    rows = []
    with open(filepath) as f:
        for line in f:
            parts = line.split()
            # Skip blank lines or lines with too few fields
            if len(parts) < 11:
                continue

            # Parse date/time components
            yr  = int(parts[0])
            mo  = int(parts[1])
            dy  = int(parts[2])
            hr  = int(parts[3])
            mn  = int(parts[4])
            sec = float(parts[5])

            # Split seconds into integer + microseconds
            # e.g. 49.830 → sec_int=49, microsec=830000
            sec_int  = int(sec)
            microsec = int(round((sec - sec_int) * 1_000_000))

            # Clamp microseconds to valid range (handles floating-point edge cases)
            microsec = min(microsec, 999_999)

            t = datetime(yr, mo, dy, hr, mn, sec_int, microsec)

            # Parse spatial and magnitude fields
            lat      = float(parts[6])
            lon      = float(parts[7])
            depth    = float(parts[8])
            mag      = float(parts[9])
            event_id = int(parts[10])

            rows.append({
                'event_id': event_id,
                'time'    : t,
                'lat'     : lat,
                'lon'     : lon,
                'depth'   : depth,
                'mag'     : mag,
            })

    df = pd.DataFrame(rows)
    # Cast types to save memory
    df['event_id'] = df['event_id'].astype('int64')
    df['lat']      = df['lat'].astype('float64')
    df['lon']      = df['lon'].astype('float64')
    df['depth']    = df['depth'].astype('float32')
    df['mag']      = df['mag'].astype('float32')
    return df


print('parse_loc_file() defined.')

In [ ]:
# ── Parse both location files ─────────────────────────────────────────────────
print('Parsing Y1 loc file...')
loc_y1 = parse_loc_file(LOC_Y1)
loc_y1['year'] = 'Y1'
print(f'  Y1: {len(loc_y1):,} events')

print('Parsing Y2 loc file...')
loc_y2 = parse_loc_file(LOC_Y2)
loc_y2['year'] = 'Y2'
print(f'  Y2: {len(loc_y2):,} events')

# Concatenate into a single DataFrame.
# ignore_index=True gives fresh 0-based row numbers.
catalog = pd.concat([loc_y1, loc_y2], ignore_index=True)
print(f'\nCombined catalog: {len(catalog):,} events')
print()

# Show a summary of the DataFrame — dtypes, non-null counts, memory usage
catalog.info()
print()
catalog.head(3)

## Step 2: Read Phase Pick Files (`.pha`)

The `.pha` file uses **HypoDD format** — a block-based format where each earthquake occupies several lines:

```
# 2022  9  1  9 35 49.830  45.955624 -129.990906   1.818   -1.39  ...  224400000  0 dd
OOAXAS1       0.773  0.554    P
OOAXAS1       1.548  0.642    S
OOAXAS2       0.843  0.817    P
...
# 2022  9  2  ...
```

**Block structure:**
- A line starting with `#` begins a new earthquake. The last token on this line is the `EventID`.
- Each following line (until the next `#`) is one phase pick:
  - `OOAXAS1` → network `OO` + station `AXAS1`
  - Second column: arrival time in **seconds relative to the origin**
  - Third column: pick weight (higher = more reliable)
  - Fourth column: phase type (`P` or `S`)

**Station name mapping:**  
The `.pha` file uses the full SEED name (e.g. `AXAS1`), but our pipeline uses short names (e.g. `AS1`).  
We define a `sta_map` dictionary to convert between the two.

**Output format:**  
We return one row per event, with columns `DDt_AS1`, `DDt_AS2`, ... for P picks and `DDSt_AS1`, `DDSt_AS2`, ... for S picks.  
`NaN` means no pick was recorded at that station for that event.

In [ ]:
# Mapping from the raw SEED station name (as it appears in the .pha file)
# to the short name used throughout the FM7 pipeline.
# The .pha lines look like "OOAXAS1" — network is "OO", SEED station is "AXAS1".
# We strip the leading network code (OO or 2F) to get the SEED name, then map it.
STA_MAP = {
    'AXAS1': 'AS1', 'AXAS2': 'AS2', 'AXCC1': 'CC1',
    'AXEC1': 'EC1', 'AXEC2': 'EC2', 'AXEC3': 'EC3', 'AXID1': 'ID1',
    'AX01A': '01A', 'AX02A': '02A', 'AX03A': '03A', 'AX04A': '04A',
    'AX05A': '05A', 'AX06A': '06A', 'AX07A': '07A', 'AX08A': '08A',
    'AX09A': '09A', 'AX10A': '10A', 'AX11A': '11A', 'AX12A': '12A',
    'AX13A': '13A', 'AX14A': '14A',
}


def parse_pha_file(filepath):
    """
    Parse a HypoDD-format .pha phase file.

    Returns
    -------
    pd.DataFrame with columns:
        event_id (int64),
        DDt_AS1 ... DDt_14A   (float32)  P arrival time relative to origin (s)
        DDSt_AS1 ... DDSt_14A (float32)  S arrival time relative to origin (s)
        nP (int16)  number of P picks for this event
        nS (int16)  number of S picks for this event
    """
    # rows will be a dict keyed by event_id.
    # Each value is a dict with the P/S pick columns.
    rows = {}
    current_id = None

    with open(filepath) as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith('#'):
                # ── Event header line ─────────────────────────────────────────
                # Example:
                #   # 2022  9  1  9 35 49.830  45.955624 -129.990906 ... 224400000  0 dd
                # The event ID is the 15th token (index 14, after stripping '#').
                parts = line[1:].split()   # remove the '#' then split
                try:
                    current_id = int(parts[14])   # EventID field
                except (IndexError, ValueError):
                    current_id = None
                    continue
                rows[current_id] = {}

            elif current_id is not None and len(line) > 5:
                # ── Phase pick line ───────────────────────────────────────────
                # Example: "OOAXAS1       0.773  0.554    P"
                parts = line.split()
                if len(parts) < 4:
                    continue

                raw_sta = parts[0]          # e.g. "OOAXAS1" or "2FAXAS1"

                # Strip leading network code to get the SEED station name.
                # Legacy stations: 2-char network code (OO) → strip first 2 chars
                # Main array:      2-char network code (2F) → also 2 chars
                seed_sta = raw_sta[2:]      # e.g. "AXAS1" or "AX01A"

                short = STA_MAP.get(seed_sta)
                if short is None:
                    continue    # unknown station — skip

                try:
                    arrival_time = float(parts[1])
                    phase        = parts[3]         # 'P' or 'S'
                except (ValueError, IndexError):
                    continue

                if phase == 'P':
                    rows[current_id][f'DDt_{short}'] = arrival_time
                elif phase == 'S':
                    rows[current_id][f'DDSt_{short}'] = arrival_time

    # Convert to DataFrame
    df = pd.DataFrame.from_dict(rows, orient='index')
    df.index.name = 'event_id'
    df = df.reset_index()
    df['event_id'] = df['event_id'].astype('int64')

    # Count picks per event
    p_cols  = [c for c in df.columns if c.startswith('DDt_')]
    s_cols  = [c for c in df.columns if c.startswith('DDSt_')]
    df['nP'] = df[p_cols].notna().sum(axis=1).astype('int16')
    df['nS'] = df[s_cols].notna().sum(axis=1).astype('int16')

    # Downcast pick columns to float32 to save memory
    for col in p_cols + s_cols:
        df[col] = df[col].astype('float32')

    return df


print('parse_pha_file() defined.')

In [ ]:
# ── Parse both phase files ────────────────────────────────────────────────────
print('Parsing Y1 phase file (this may take ~30 s for large files)...')
pha_y1 = parse_pha_file(PHA_Y1)
pha_y1['year'] = 'Y1'
print(f'  Y1: {len(pha_y1):,} events')
print(f'  Median P picks per event: {pha_y1["nP"].median():.0f}')
print(f'  Median S picks per event: {pha_y1["nS"].median():.0f}')

print('\nParsing Y2 phase file...')
pha_y2 = parse_pha_file(PHA_Y2)
pha_y2['year'] = 'Y2'
print(f'  Y2: {len(pha_y2):,} events')
print(f'  Median P picks per event: {pha_y2["nP"].median():.0f}')
print(f'  Median S picks per event: {pha_y2["nS"].median():.0f}')

# Combine Y1 and Y2
phases = pd.concat([pha_y1, pha_y2], ignore_index=True)
print(f'\nCombined phases: {len(phases):,} rows')
print(f'  Columns: {phases.columns.tolist()[:6]} ...  ({phases.shape[1]} total)')

## Step 3: Validate & Save

Before saving, we run a few sanity checks:

1. **Event counts** — do we see roughly the expected numbers (6,727 Y1 + 48,244 Y2)?
2. **Geographic bounds** — all Axial Seamount events should fall in:
   - Latitude: 45.8 – 46.1 °N
   - Longitude: -130.1 – -129.9 °E
3. **Depth range** — typically 0 – 10 km for this network
4. **Phase coverage** — every event in the catalog should have at least one phase pick

Any events that fail these checks are flagged (not removed), so you can inspect them.

In [ ]:
# ── Validation checks ─────────────────────────────────────────────────────────
print('=== Catalog Validation ===')

n_y1 = (catalog['year'] == 'Y1').sum()
n_y2 = (catalog['year'] == 'Y2').sum()
print(f'  Y1 events : {n_y1:>7,}   (expected ~6,727)')
print(f'  Y2 events : {n_y2:>7,}   (expected ~48,244)')
print(f'  Total     : {len(catalog):>7,}')

# Geographic bounds check
LAT_MIN, LAT_MAX = 45.8, 46.1
LON_MIN, LON_MAX = -130.1, -129.9
out_lat = catalog[(catalog['lat'] < LAT_MIN) | (catalog['lat'] > LAT_MAX)]
out_lon = catalog[(catalog['lon'] < LON_MIN) | (catalog['lon'] > LON_MAX)]
print(f'\n  Events outside lat [{LAT_MIN}, {LAT_MAX}]: {len(out_lat)}')
print(f'  Events outside lon [{LON_MIN}, {LON_MAX}]: {len(out_lon)}')
if len(out_lat) > 0 or len(out_lon) > 0:
    print('  WARNING: These events may have bad locations. Review them.')

# Depth range check
print(f'\n  Depth range: {catalog["depth"].min():.2f} – {catalog["depth"].max():.2f} km')
neg_depth = (catalog['depth'] < 0).sum()
print(f'  Events with negative depth: {neg_depth}')

# Phase coverage check
catalog_ids = set(catalog['event_id'])
phases_ids  = set(phases['event_id'])
missing_phases = catalog_ids - phases_ids
extra_phases   = phases_ids  - catalog_ids
print(f'\n  Catalog events missing from phases file : {len(missing_phases)}')
print(f'  Phase events missing from catalog       : {len(extra_phases)}')

print('\n=== Descriptive Statistics ===')
print(catalog[['lat', 'lon', 'depth', 'mag']].describe().round(3))

In [ ]:
# ── Save to Parquet ───────────────────────────────────────────────────────────
# Parquet is a columnar binary format. It's much faster to read than CSV
# and preserves exact data types (no silent float→string conversions).
# The 'pyarrow' engine is recommended and comes with pandas >= 1.3.

print(f'Saving catalog → {CATALOG_PQ}')
catalog.to_parquet(CATALOG_PQ, engine='pyarrow', index=False)
sz = CATALOG_PQ.stat().st_size / 1e6
print(f'  Saved {len(catalog):,} rows, {sz:.2f} MB')

print(f'\nSaving phases  → {PHASES_PQ}')
phases.to_parquet(PHASES_PQ, engine='pyarrow', index=False)
sz = PHASES_PQ.stat().st_size / 1e6
print(f'  Saved {len(phases):,} rows, {sz:.2f} MB')

# Quick round-trip check — reload and verify row count
catalog_check = pd.read_parquet(CATALOG_PQ)
phases_check  = pd.read_parquet(PHASES_PQ)
assert len(catalog_check) == len(catalog), 'Catalog row count mismatch after save!'
assert len(phases_check)  == len(phases),  'Phases row count mismatch after save!'
print('\nRound-trip check passed.')

## Step 4: Visualizations

Four diagnostic plots help us verify the catalog looks sensible:

1. **Map** — spatial distribution of events (Y1=blue, Y2=orange) with station triangles
2. **Depth histogram** — depth distribution (axis flipped so deeper is down, like the real Earth)
3. **Time series** — monthly event count stacked bar chart showing Y1 and Y2
4. **Phase coverage heatmap** — rows=stations, columns=months, cell=number of P picks

All plots are saved as PNG files to `EMAIL_PLOTS_DIR` so the email function can attach them.

In [ ]:
# ── Helper: save a figure ─────────────────────────────────────────────────────
def savefig(fig, name):
    """Save figure to EMAIL_PLOTS_DIR/<name>.png at 150 dpi."""
    out = EMAIL_PLOTS_DIR / f'{name}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved → {out}')
    return str(out)


saved_plots = []

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 1: MAP — event locations coloured by year, station triangles
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 7))

# Plot Y1 events first (they'll be underneath Y2)
cat_y1_mask = catalog['year'] == 'Y1'
cat_y2_mask = catalog['year'] == 'Y2'

ax.scatter(
    catalog.loc[cat_y1_mask, 'lon'],
    catalog.loc[cat_y1_mask, 'lat'],
    s=1, c='steelblue', alpha=0.4, label=f'Y1 (n={cat_y1_mask.sum():,})', rasterized=True
)
ax.scatter(
    catalog.loc[cat_y2_mask, 'lon'],
    catalog.loc[cat_y2_mask, 'lat'],
    s=1, c='darkorange', alpha=0.3, label=f'Y2 (n={cat_y2_mask.sum():,})', rasterized=True
)

# Overlay station locations as triangles
for sta, (slat, slon, _) in STATION_COORDS.items():
    ax.plot(slon, slat, marker='^', ms=7, color='black', zorder=5)
    ax.annotate(sta, (slon, slat), textcoords='offset points',
                xytext=(4, 3), fontsize=5, color='black')

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Axial Seamount Seismicity — Y1+Y2 Combined\n(blue=Y1, orange=Y2, triangles=stations)')
ax.legend(loc='upper right', markerscale=5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3, lw=0.5)
saved_plots.append(savefig(fig, 'step01_map'))


# ══════════════════════════════════════════════════════════════════════════════
# PLOT 2: DEPTH HISTOGRAM — separate Y1 and Y2
# Flip y-axis so depth increases downward (like looking into the Earth)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(6, 5))

bins = np.arange(0, catalog['depth'].max() + 0.5, 0.25)
ax.hist(catalog.loc[cat_y1_mask, 'depth'], bins=bins, orientation='horizontal',
        color='steelblue', alpha=0.7, label='Y1')
ax.hist(catalog.loc[cat_y2_mask, 'depth'], bins=bins, orientation='horizontal',
        color='darkorange', alpha=0.5, label='Y2')

ax.invert_yaxis()   # flip so depth 0 is at top, deep at bottom
ax.set_xlabel('Number of events')
ax.set_ylabel('Depth (km)')
ax.set_title('Depth Distribution')
ax.legend()
ax.grid(True, axis='x', alpha=0.3)
saved_plots.append(savefig(fig, 'step01_depth'))


# ══════════════════════════════════════════════════════════════════════════════
# PLOT 3: TIME SERIES — monthly event count, stacked bars
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 4))

# Create a 'month' column in YYYY-MM format for grouping
catalog_ts = catalog.copy()
catalog_ts['month'] = pd.to_datetime(catalog_ts['time']).dt.to_period('M')

monthly_y1 = catalog_ts[catalog_ts['year'] == 'Y1'].groupby('month').size()
monthly_y2 = catalog_ts[catalog_ts['year'] == 'Y2'].groupby('month').size()

# Align both series to a common monthly index
all_months = monthly_y1.index.union(monthly_y2.index)
monthly_y1 = monthly_y1.reindex(all_months, fill_value=0)
monthly_y2 = monthly_y2.reindex(all_months, fill_value=0)

x = np.arange(len(all_months))
ax.bar(x, monthly_y1.values, color='steelblue', label='Y1', width=0.8)
ax.bar(x, monthly_y2.values, bottom=monthly_y1.values,
       color='darkorange', label='Y2', width=0.8)

# Tick labels: show every 3rd month to avoid crowding
tick_step = max(1, len(all_months) // 10)
ax.set_xticks(x[::tick_step])
ax.set_xticklabels([str(m) for m in all_months[::tick_step]], rotation=45, ha='right')
ax.set_ylabel('Events per month')
ax.set_title('Monthly Event Count — Y1+Y2')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
saved_plots.append(savefig(fig, 'step01_timeseries'))


# ══════════════════════════════════════════════════════════════════════════════
# PLOT 4: PHASE COVERAGE HEATMAP — rows=stations, columns=months
# Value in each cell = number of P picks for that station in that month
# ══════════════════════════════════════════════════════════════════════════════
# Join phases with catalog to get time info
phases_ts = phases.merge(catalog[['event_id', 'time']], on='event_id', how='left')
phases_ts['month'] = pd.to_datetime(phases_ts['time']).dt.to_period('M')

# Build a count matrix: for each station, how many P picks exist per month?
p_cols = [c for c in phases.columns if c.startswith('DDt_')]
cov_data = {}
for col in p_cols:
    sta = col.replace('DDt_', '')   # e.g. 'DDt_AS1' → 'AS1'
    # A non-NaN value = a P pick exists for this event at this station
    monthly_picks = (
        phases_ts[phases_ts[col].notna()]
        .groupby('month').size()
        .reindex(all_months, fill_value=0)
    )
    cov_data[sta] = monthly_picks.values

cov_df = pd.DataFrame(cov_data, index=[str(m) for m in all_months]).T
# Reorder rows to match STATIONS_ALL order
cov_df = cov_df.reindex([s for s in STATIONS_ALL if s in cov_df.index])

fig, ax = plt.subplots(figsize=(max(10, len(all_months) * 0.45), 7))

if HAS_SNS:
    sns.heatmap(
        cov_df, ax=ax,
        cmap='YlOrRd', linewidths=0.3,
        cbar_kws={'label': 'P picks per month'},
        xticklabels=tick_step,
    )
else:
    # Fallback: matplotlib imshow
    im = ax.imshow(cov_df.values, aspect='auto', cmap='YlOrRd')
    ax.set_yticks(range(len(cov_df)))
    ax.set_yticklabels(cov_df.index)
    ax.set_xticks(range(0, len(all_months), tick_step))
    ax.set_xticklabels([str(m) for m in all_months[::tick_step]], rotation=45, ha='right')
    fig.colorbar(im, ax=ax, label='P picks per month')

ax.set_title('Phase Coverage Heatmap\n(rows = stations, columns = months, color = P pick count)')
ax.set_xlabel('Month')
ax.set_ylabel('Station')
saved_plots.append(savefig(fig, 'step01_coverage'))

print(f'\nAll 4 plots saved to {EMAIL_PLOTS_DIR}')

In [ ]:
# ── Send step completion email ────────────────────────────────────────────────
# The step_email() function (from email_utils.py) packages up a summary dict
# and the plot file paths and sends them via macOS Mail.app.
# If you're on a non-Mac machine, this will print a warning but won't crash.

stats = {
    'Y1 events'             : f'{n_y1:,}',
    'Y2 events'             : f'{n_y2:,}',
    'Total events'          : f'{len(catalog):,}',
    'Total phase rows'      : f'{len(phases):,}',
    'Events outside bounds' : len(out_lat) + len(out_lon),
    'Events missing phases' : len(missing_phases),
    'Catalog saved to'      : str(CATALOG_PQ),
    'Phases saved to'       : str(PHASES_PQ),
}

try:
    step_email(
        step_num=1,
        step_name='Parse mldd Catalog',
        stats=stats,
        plots=saved_plots,
    )
except Exception as e:
    print(f'[email skipped] {e}')

print('\nNotebook 01 complete.')